# 06b — LoRA vs QLoRA Comparison

**Phase 6b** of the FYP: head-to-head delta table between Phase 5 (LoRA, fp16) and Phase 6a (QLoRA, 4-bit).

**Goal:** quantify the cost of 4-bit quantisation across four axes:
- **Accuracy delta** — does QLoRA hurt task performance?
- **Memory delta** — how much GPU RAM does QLoRA save?
- **Training time delta** — is QLoRA slower (extra dequantisation steps)?
- **Inference latency** — how fast can each adapter score molecules at test time?

**Inputs:** `lora_results.csv` (Phase 5, 18 rows) and `qlora_results.csv` (Phase 6a, 9 rows).

**Outputs:** `lora_vs_qlora.csv` (9 rows, one per model×dataset) saved to Drive + GitHub.

Self-contained: survives Colab VM resets.

## 1. Environment + Drive mount

In [ ]:
%pip install -q --upgrade torchao
%pip install -q --upgrade transformers peft accelerate bitsandbytes rdkit pandas scikit-learn

In [ ]:
import os, sys, time, json, gc, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/chemistry-peft-fyp')
RESULTS_DIR  = PROJECT_ROOT / 'results'
ADAPTERS_DIR = PROJECT_ROOT / 'adapters'
SPLITS_DIR   = PROJECT_ROOT / 'data' / 'splits'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LORA_CSV     = RESULTS_DIR / 'lora_results.csv'
QLORA_CSV    = RESULTS_DIR / 'qlora_results.csv'
COMPARE_CSV  = RESULTS_DIR / 'lora_vs_qlora.csv'

for p in (LORA_CSV, QLORA_CSV):
    assert p.exists(), f'Missing: {p}'
    print('found:', p)

## 2. Load Phase 5 + Phase 6a CSVs

In [ ]:
lora_df  = pd.read_csv(LORA_CSV)
qlora_df = pd.read_csv(QLORA_CSV)

print('LoRA rows: ', len(lora_df))
print('QLoRA rows:', len(qlora_df))
print()
print('--- LoRA preview ---')
print(lora_df[['model','dataset','value_primary','peak_mem_mb','train_time_s']].to_string(index=False))
print()
print('--- QLoRA preview ---')
print(qlora_df[['model','dataset','value_primary','peak_mem_mb','train_time_s']].to_string(index=False))

## 3. Pick the winning LoRA row per (model, dataset)

Phase 5 ran a small sweep (multiple rank/lr combinations), so we need to pick the best LoRA row per (model, dataset) before comparing.

- **Classification (BBBP, BACE)** — higher `value_primary` (roc_auc) wins.
- **Regression (ESOL)** — lower `value_primary` (rmse) wins.

In [ ]:
def best_lora_row(df):
    rows = []
    for (model, dataset), group in df.groupby(['model','dataset']):
        task = group['task'].iloc[0]
        if task == 'classification':
            row = group.loc[group['value_primary'].idxmax()]
        else:
            row = group.loc[group['value_primary'].idxmin()]
        rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)

lora_best = best_lora_row(lora_df)
print('Best LoRA per (model, dataset):')
print(lora_best[['model','dataset','lora_rank','lora_lr','value_primary','peak_mem_mb','train_time_s']].to_string(index=False))

## 4. Build the delta table

Join LoRA best + QLoRA on `(model, dataset)` and compute deltas.

**Delta convention:** `QLoRA - LoRA`.
- For roc_auc (BBBP, BACE): positive delta = QLoRA better.
- For rmse (ESOL): negative delta = QLoRA better.
- For memory and time: negative delta = QLoRA cheaper.

In [ ]:
merged = lora_best.merge(
    qlora_df,
    on=['model','dataset','task'],
    suffixes=('_lora','_qlora')
)

merged['delta_primary']  = merged['value_primary_qlora']  - merged['value_primary_lora']
merged['delta_mem_mb']   = merged['peak_mem_mb_qlora']    - merged['peak_mem_mb_lora']
merged['delta_time_s']   = merged['train_time_s_qlora']   - merged['train_time_s_lora']
merged['mem_saving_pct'] = (merged['delta_mem_mb'] / merged['peak_mem_mb_lora']) * 100

compare = merged[[
    'model','dataset','task','metric_primary_lora',
    'value_primary_lora','value_primary_qlora','delta_primary',
    'peak_mem_mb_lora','peak_mem_mb_qlora','delta_mem_mb','mem_saving_pct',
    'train_time_s_lora','train_time_s_qlora','delta_time_s',
]].rename(columns={'metric_primary_lora':'metric_primary'})

print('--- LoRA vs QLoRA delta table ---')
print(compare.to_string(index=False))

## 5. Inference latency benchmark

Training time tells us how long fine-tuning takes. **Inference latency** matters more for downstream use: once the adapter is trained, how fast can it score new molecules?

We load each saved adapter on top of its base model (LoRA = fp16, QLoRA = 4-bit) and time scoring on a small batch from the test split.

**Note:** if any adapter directory is missing, the run is skipped and reported.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

MODEL_CONFIGS = {
    'microsoft/Phi-4-mini-instruct': {'short': 'phi4'},
    'HuggingFaceTB/SmolLM3-3B':      {'short': 'smollm3'},
    'gpt2':                          {'short': 'gpt2'},
}

PROMPT_TEMPLATE = (
    'You are a chemistry assistant. Given the SMILES, answer yes or no.\n'
    'SMILES: {smiles}\nAnswer:'
)

In [ ]:
def load_test_smiles(dataset_name, n=20):
    split_path = SPLITS_DIR / dataset_name / 'test.csv'
    if not split_path.exists():
        print(f'  test split missing: {split_path}')
        return []
    df = pd.read_csv(split_path)
    smiles_col = 'smiles' if 'smiles' in df.columns else df.columns[0]
    return df[smiles_col].head(n).tolist()

def time_inference(base_model, adapter_dir, smiles_list, tokenizer, device):
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    prompts = [PROMPT_TEMPLATE.format(smiles=s) for s in smiles_list]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for p in prompts:
            enc = tokenizer(p, return_tensors='pt', truncation=True, max_length=256).to(device)
            _ = model(**enc)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - t0
    return elapsed / len(prompts) * 1000.0

def adapter_path(model_name, dataset, mode):
    short = MODEL_CONFIGS[model_name]['short']
    return ADAPTERS_DIR / mode / short / dataset

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
latency_rows = []

for model_name in MODEL_CONFIGS:
    print(f'\n=== {model_name} ===')
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    for dataset in ['BBBP','BACE','ESOL']:
        smiles_list = load_test_smiles(dataset)
        if not smiles_list:
            continue

        row = {'model': model_name, 'dataset': dataset}

        lora_dir = adapter_path(model_name, dataset, 'lora')
        if lora_dir.exists():
            base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, trust_remote_code=True).to(device)
            row['lora_latency_ms'] = time_inference(base, str(lora_dir), smiles_list, tokenizer, device)
            del base; gc.collect(); torch.cuda.empty_cache()
        else:
            row['lora_latency_ms'] = float('nan')
            print(f'  skip LoRA {dataset}: {lora_dir} missing')

        qlora_dir = adapter_path(model_name, dataset, 'qlora')
        if qlora_dir.exists():
            base = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=BNB_CONFIG, trust_remote_code=True)
            row['qlora_latency_ms'] = time_inference(base, str(qlora_dir), smiles_list, tokenizer, device)
            del base; gc.collect(); torch.cuda.empty_cache()
        else:
            row['qlora_latency_ms'] = float('nan')
            print(f'  skip QLoRA {dataset}: {qlora_dir} missing')

        if not math.isnan(row['lora_latency_ms']) and not math.isnan(row['qlora_latency_ms']):
            row['delta_latency_ms'] = row['qlora_latency_ms'] - row['lora_latency_ms']
        else:
            row['delta_latency_ms'] = float('nan')

        latency_rows.append(row)
        print(f'  {dataset}: LoRA {row["lora_latency_ms"]:.1f} ms | QLoRA {row["qlora_latency_ms"]:.1f} ms')

latency_df = pd.DataFrame(latency_rows)
print('\n--- Inference latency (ms / molecule) ---')
print(latency_df.to_string(index=False))

## 6. Merge latency into the delta table + save

In [ ]:
if not latency_df.empty:
    final = compare.merge(latency_df, on=['model','dataset'], how='left')
else:
    final = compare.copy()
    final['lora_latency_ms']  = float('nan')
    final['qlora_latency_ms'] = float('nan')
    final['delta_latency_ms'] = float('nan')

final.to_csv(COMPARE_CSV, index=False)
print(f'\nSaved: {COMPARE_CSV}  ({len(final)} rows)')
print()
print(final.to_string(index=False))

## 7. Quick summary

Average savings (memory %, time s) and accuracy hit across the 9 (model, dataset) pairs.

In [ ]:
summary = {
    'avg_mem_saving_pct': final['mem_saving_pct'].mean(),
    'avg_train_time_delta_s': final['delta_time_s'].mean(),
    'avg_accuracy_delta_classification': final.loc[final['task']=='classification', 'delta_primary'].mean(),
    'avg_rmse_delta_regression': final.loc[final['task']=='regression', 'delta_primary'].mean(),
}
if 'delta_latency_ms' in final.columns:
    summary['avg_latency_delta_ms'] = final['delta_latency_ms'].mean()

print('--- Phase 6 summary ---')
for k, v in summary.items():
    print(f'  {k}: {v:.3f}')